## Análise

Neste notebook respondo às 4 perguntas de negócio consultando o modelo dimensional da camada Gold. Todas as consultas cruzam fatos e dimensões e usam o período 2021–2025.

### Pergunta 1: há concentração de mercado por empresa?

Olho a concentração por duas óticas: o **prêmio**, que mede a força comercial no ano (fluxo), e a **provisão**, que mede o volume de obrigações com os titulares de títulos (saldo). Em cada uma, calculo por ano a participação da maior empresa (top 1), a soma das 3 e das 5 maiores, e o **HHI** (Índice Herfindahl-Hirschman), que é a soma dos quadrados das participações em %. Pelo critério do CADE, HHI abaixo de 1.500 indica mercado não concentrado, entre 1.500 e 2.500 moderadamente concentrado, e acima de 2.500 altamente concentrado.

#### 1.1 Prêmios

In [0]:
%sql
WITH anual AS (
  SELECT t.ano, f.coenti, e.nome_empresa, SUM(f.premio) AS premio
  FROM susep_capitalizacao.gold.fato_premios_uf f
  JOIN susep_capitalizacao.gold.dim_tempo   t ON t.damesano = f.damesano
  JOIN susep_capitalizacao.gold.dim_empresa e ON e.coenti   = f.coenti      -- novo: nome da empresa
  GROUP BY t.ano, f.coenti, e.nome_empresa
  HAVING SUM(f.premio) <> 0   
),
participacao AS (
  SELECT
    ano, coenti, nome_empresa, premio,
    100 * premio / SUM(premio) OVER (PARTITION BY ano)        AS pct,
    ROW_NUMBER() OVER (PARTITION BY ano ORDER BY premio DESC) AS posicao
  FROM anual
)
SELECT
  ano,
  COUNT(*)                                             AS empresas,
  ROUND(SUM(premio) / 1e9, 2)                          AS premio_total_bi,
  MAX(CASE WHEN posicao = 1 THEN nome_empresa END)     AS lider,                -- novo
  ROUND(SUM(CASE WHEN posicao = 1  THEN pct END), 1)   AS top1_pct,
  ROUND(SUM(CASE WHEN posicao <= 3 THEN pct END), 1)   AS top3_pct,
  ROUND(SUM(CASE WHEN posicao <= 5 THEN pct END), 1)   AS top5_pct,
  ROUND(SUM(POWER(pct, 2)), 0)                         AS hhi,
  CASE
    WHEN SUM(POWER(pct, 2)) < 1500  THEN 'Não concentrado'
    WHEN SUM(POWER(pct, 2)) <= 2500 THEN 'Moderadamente concentrado'
    ELSE 'Altamente concentrado'
  END                                                  AS classificacao
FROM participacao
GROUP BY ano
ORDER BY ano

O gráfico da esquerda mostra a participação das maiores empresas, com o nome da líder de cada ano. O da direita posiciona o HHI nas faixas do CADE. O código do gráfico fica numa função, reaproveitada na ótica da provisão.

In [0]:
import matplotlib.pyplot as plt

def grafico_concentracao(res, medida, titulo):
    """Desenha participação das maiores empresas (com a líder de cada ano) e HHI nas faixas do CADE."""
    res  = res.sort_values("ano")
    anos = res["ano"].astype(int)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    fig.patch.set_facecolor("#fcfcfb")

    # Esquerda: participação das maiores empresas
    series = [("top1_pct", "Maior empresa", "#2a78d6"),
              ("top3_pct", "3 maiores",     "#eb6834"),
              ("top5_pct", "5 maiores",     "#1baf7a")]
    for coluna, rotulo, cor in series:
        y = res[coluna].astype(float)
        ax1.plot(anos, y, color=cor, linewidth=2, marker="o", markersize=6, label=rotulo)
        ax1.annotate(f"{y.iloc[-1]:.0f}%", (anos.iloc[-1], y.iloc[-1]),
                     xytext=(8, 0), textcoords="offset points", va="center", fontsize=9, color="#333")
    for ano, pct, nome in zip(anos, res["top1_pct"].astype(float), res["lider"]):
        ax1.annotate(nome.split()[0].title(), (ano, pct),
                     xytext=(0, -14), textcoords="offset points", ha="center", fontsize=8, color="#555")
    ax1.set_ylim(0, 100)
    ax1.set_title(f"Participação no {medida} (%)", loc="left", fontsize=11)
    ax1.legend(frameon=False, loc="center left", bbox_to_anchor=(0, 0.40), fontsize=9)

    # Direita: HHI nas faixas do CADE
    hhi = res["hhi"].astype(float)
    for inicio, fim, cor, rotulo in [(0, 1500, "#f0f0ee", "Não concentrado"),
                                     (1500, 2500, "#e2e1dc", "Moderadamente concentrado"),
                                     (2500, 3500, "#d3d2cb", "Altamente concentrado")]:
        ax2.axhspan(inicio, fim, color=cor, zorder=0)
        ax2.text(anos.iloc[0] - 0.35, inicio + 60, rotulo, fontsize=8, color="#555", va="bottom")
    ax2.plot(anos, hhi, color="#2a78d6", linewidth=2, marker="o", markersize=6)
    ax2.annotate(f"{hhi.iloc[-1]:.0f}", (anos.iloc[-1], hhi.iloc[-1]),
                 xytext=(8, 0), textcoords="offset points", va="center", fontsize=9, color="#333")
    ax2.set_ylim(0, 3500)
    ax2.set_title("HHI por ano (faixas do CADE)", loc="left", fontsize=11)

    for ax in (ax1, ax2):
        ax.set_facecolor("#fcfcfb")
        ax.set_xticks(anos)
        ax.set_xlim(anos.min() - 0.4, anos.max() + 0.5)
        ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(titulo, x=0.01, ha="left", fontsize=12)
    plt.tight_layout()
    plt.show()

res_premio = _sqldf.toPandas()
grafico_concentracao(res_premio, "prêmio total",
                     "Pergunta 1: concentração por empresa em prêmios, 2021–2025")

#### 1.2 Provisões

Repito a análise com o saldo de provisão. Como a provisão é semiaditiva, uso o saldo de dezembro de cada ano.

In [0]:
%sql
WITH anual AS (                       -- saldo de dezembro de cada empresa
  SELECT t.ano, f.coenti, e.nome_empresa, f.provisao_total AS valor
  FROM susep_capitalizacao.gold.fato_provisao f
  JOIN susep_capitalizacao.gold.dim_tempo   t ON t.damesano = f.damesano
  JOIN susep_capitalizacao.gold.dim_empresa e ON e.coenti   = f.coenti
  WHERE t.mes = 12
  AND f.provisao_total > 0           
),
participacao AS (
  SELECT
    ano, coenti, nome_empresa, valor,
    100 * valor / SUM(valor) OVER (PARTITION BY ano)        AS pct,
    ROW_NUMBER() OVER (PARTITION BY ano ORDER BY valor DESC) AS posicao
  FROM anual
)
SELECT
  ano,
  COUNT(*)                                             AS empresas,
  ROUND(SUM(valor) / 1e9, 2)                           AS provisao_total_bi,
  MAX(CASE WHEN posicao = 1 THEN nome_empresa END)     AS lider,
  ROUND(SUM(CASE WHEN posicao = 1  THEN pct END), 1)   AS top1_pct,
  ROUND(SUM(CASE WHEN posicao <= 3 THEN pct END), 1)   AS top3_pct,
  ROUND(SUM(CASE WHEN posicao <= 5 THEN pct END), 1)   AS top5_pct,
  ROUND(SUM(POWER(pct, 2)), 0)                         AS hhi,
  CASE
    WHEN SUM(POWER(pct, 2)) < 1500  THEN 'Não concentrado'
    WHEN SUM(POWER(pct, 2)) <= 2500 THEN 'Moderadamente concentrado'
    ELSE 'Altamente concentrado'
  END                                                  AS classificacao
FROM participacao
GROUP BY ano
ORDER BY ano

In [0]:
res_provisao = _sqldf.toPandas()
grafico_concentracao(res_provisao, "saldo de provisão em dezembro",
                     "Pergunta 1: concentração por empresa em provisões, 2021–2025")

#### Discussão da pergunta 1

| Ótica | Top 1 | Top 3 | Top 5 | HHI | Classificação (CADE) |
|---|---|---|---|---|---|
| Prêmios (fluxo do ano) | 22–23% | 55–58% | 70–75% | 1.307–1.396 | Não concentrado |
| Provisões (saldo em dezembro) | 25–29% | 61–66% | 79–83% | 1.614–1.743 | Moderadamente concentrado |

**Há concentração, e o grau depende da ótica.** Em prêmios, o mercado não é concentrado pelo HHI, mas a liderança é compartilhada: as 3 maiores empresas respondem por mais da metade das vendas. Nenhuma empresa domina sozinha (a maior tem cerca de 22%). Em provisões, a concentração é maior e o mercado é moderadamente concentrado: as líderes detêm uma fatia maior das obrigações com os titulares do que das vendas do ano.

**Liderança.** Em provisões, o Bradesco liderou em 2021 e a Brasilcap de 2022 a 2025. Em prêmios, o Bradesco liderou em todos os anos, exceto 2023, quando a Brasilcap ficou à frente.

**Evolução.** Nas duas óticas, a concentração atinge o pico em 2022–2023 e recua levemente depois: o HHI de prêmios cai de 1.396 (2022) para 1.307 (2025), e o de provisões de 1.743 (2023) para 1.614 (2025). No mesmo período, o mercado cresceu: o prêmio anual passou de R$ 24,3 bi para R$ 33,4 bi, e o saldo de provisões de R$ 33,2 bi para R$ 44,2 bi.

**Por que as óticas divergem.** O prêmio mede o que foi vendido no ano, e a provisão, o que se acumulou ao longo do tempo. As líderes captam volumes maiores ano após ano, e nem todo o saldo sai: parte dos titulares mantém os títulos ou não resgata os valores. Assim, o saldo das maiores companhias cresce mais que o das demais, e a concentração que o fluxo de um único ano não evidencia aparece na provisão.

### Pergunta 2: o mercado é concentrado geograficamente?

Comparo a participação de cada UF no prêmio total no primeiro e no último ano do período (2021 e 2025), com a região de cada uma. Assim vejo quais UFs concentram o mercado, quanto as regiões representam e se essa distribuição mudou.

In [0]:
import matplotlib.pyplot as plt

# 1. Consulta: prêmio por ano, UF e região
df = spark.sql("""
    SELECT t.ano, f.uf, u.regiao, SUM(f.premio) AS premio
    FROM susep_capitalizacao.gold.fato_premios_uf f
    JOIN susep_capitalizacao.gold.dim_tempo t ON t.damesano = f.damesano
    JOIN susep_capitalizacao.gold.dim_uf    u ON u.uf       = f.uf
    GROUP BY t.ano, f.uf, u.regiao
""").toPandas()
df["premio"] = df["premio"].astype(float)
df["pct"] = 100 * df["premio"] / df.groupby("ano")["premio"].transform("sum")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.1, 1]})
fig.patch.set_facecolor("#fcfcfb")

# 2. Esquerda: curva de concentração (2021 em cinza como referência, 2025 em destaque)
for ano, cor in [(2021, "#9a9a95"), (2025, "#2a78d6")]:
    d = df[df["ano"] == ano].sort_values("pct", ascending=False)
    acum = d["pct"].cumsum().values
    x = range(1, len(acum) + 1)
    ax1.plot(x, acum, color=cor, linewidth=2, marker="o", markersize=4, label=str(ano))
    if ano == 2025:
        for n in (1, 3, 5):
            ax1.annotate(f"{acum[n-1]:.0f}%", (n, acum[n-1]), xytext=(6, -12),
                         textcoords="offset points", fontsize=9, color="#333")
        top5 = ", ".join(d["uf"].head(5))

# Referência: se o prêmio fosse dividido igualmente entre as 27 UFs (novo)
ax1.plot([0, 27], [0, 100], linestyle="--", color="#bbb", linewidth=1, label="Distribuição igual")

ax1.set_ylim(0, 105)
ax1.set_xticks([1, 3, 5, 10, 15, 20, 27])
ax1.set_xlabel("Número de UFs (da maior para a menor participação)")
ax1.set_ylabel("% acumulado do prêmio")
ax1.set_title("Curva de concentração por UF", loc="left", fontsize=11)
ax1.text(27, 8, f"5 maiores em 2025: {top5}", ha="right", fontsize=9, color="#555")
ax1.legend(frameon=False, loc="center right", fontsize=9)

# 3. Direita: participação por região, ano a ano (barras 100% empilhadas)
ordem = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
cores = {"Sudeste": "#2a78d6", "Sul": "#eb6834", "Nordeste": "#1baf7a",
         "Centro-Oeste": "#eda100", "Norte": "#e87ba4"}
reg = df.groupby(["ano", "regiao"])["pct"].sum().unstack()[ordem].sort_index(ascending=False)

esquerda = [0] * len(reg)
for regiao in ordem:
    valores = reg[regiao].values
    ax2.barh(reg.index.astype(str), valores, left=esquerda, color=cores[regiao],
             edgecolor="#fcfcfb", linewidth=1.5, height=0.65, label=regiao)
    for i, v in enumerate(valores):
        if v >= 5:                                  # fatia larga: rótulo dentro
            ax2.text(esquerda[i] + v / 2, i, f"{v:.0f}%", ha="center", va="center",
                     fontsize=8, color="white")
        elif regiao == ordem[-1]:                   # Norte: fatia estreita, rótulo fora da barra (novo)
            ax2.text(100.8, i, f"{v:.0f}%", ha="left", va="center", fontsize=8, color="#333")
    esquerda = [e + v for e, v in zip(esquerda, valores)]

ax2.set_xlim(0, 106)                                # espaço à direita para o rótulo do Norte (novo)
ax2.set_xticks(range(0, 101, 20))
ax2.set_xlabel("% do prêmio do ano")
ax2.set_title("Participação por região", loc="left", fontsize=11)
ax2.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=5, fontsize=9)

for ax in (ax1, ax2):
    ax.set_facecolor("#fcfcfb")
    ax.spines[["top", "right"]].set_visible(False)
ax1.grid(color="#e5e5e5", linewidth=0.8)

fig.suptitle("Pergunta 2: concentração geográfica do prêmio, 2021–2025", x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()


In [0]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 10 maiores UFs no período; as demais viram "Demais UFs"
top10 = df.groupby("uf")["premio"].sum().nlargest(10).index
d = df.assign(grupo=df["uf"].where(df["uf"].isin(top10), "Demais UFs"))
m = d.groupby(["grupo", "ano"])["pct"].sum().unstack().loc[list(top10) + ["Demais UFs"]]
anos = m.columns.astype(int)

fig, ax = plt.subplots(figsize=(8.5, 6))
fig.patch.set_facecolor("#fcfcfb")
cmap = LinearSegmentedColormap.from_list("azul", ["#eef4fc", "#86b6ef", "#2a78d6", "#104281"])
ax.imshow(m.values, cmap=cmap, aspect="auto", vmin=0, vmax=m.values.max())

# Valor em cada célula (texto branco nas cores escuras)
for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        v = m.values[i, j]
        ax.text(j, i, f"{v:.1f}%", ha="center", va="center", fontsize=8,
                color="white" if v > 0.5 * m.values.max() else "#333")
    # Variação 2021 → 2025, em pontos percentuais
    delta = m.iloc[i][anos.max()] - m.iloc[i][anos.min()]
    ax.text(len(anos) - 0.3, i, f"{delta:+.1f} p.p.", va="center", fontsize=8, color="#333")

ax.text(len(anos) - 0.3, -0.8, "2021→2025", fontsize=8, color="#555")
ax.set_xlim(-0.5, len(anos) + 0.8)
ax.set_xticks(range(len(anos)), anos)
ax.set_yticks(range(len(m.index)), m.index)
ax.tick_params(length=0)
ax.spines[:].set_visible(False)
ax.set_title("Participação de cada UF no prêmio (%), 2021–2025", loc="left", fontsize=11, pad=18)
plt.tight_layout()
plt.show()

#### Discussão da pergunta 2

| Indicador | 2021 | 2025 |
|---|---|---|
| Maior UF (SP) | 37,7% | 37,1% |
| 5 maiores UFs | 70% | 69% |
| Sudeste | 58% | 55% |
| Sul | 19% | 19% |
| Nordeste | 10% | 12% |
| Centro-Oeste | 9% | 9% |
| Norte | 4% | 4% |

**O mercado é concentrado geograficamente.** São Paulo sozinho responde por mais de um terço do prêmio, e 5 UFs (SP, MG, RS, RJ e PR) somam cerca de 70%. Por região, o Sudeste concentra mais da metade do mercado, e Sudeste e Sul juntos passam de 70%. Essa distribuição acompanha o peso econômico das regiões, onde está a maior parte da renda e da base de clientes.

**A concentração mudou pouco, com leve dispersão.** A curva de concentração de 2025 fica ligeiramente abaixo da de 2021, e o Sudeste perdeu 3 p.p. no período. Os ganhos foram pulverizados: as UFs fora do top 10 passaram de 13,5% para 15,2% (+1,7 p.p.), o Nordeste subiu de 10% para 12%, e RS (+1,2 p.p.) e PE (+0,8 p.p.) cresceram. As maiores quedas foram de RJ (−1,3 p.p.) e DF (−0,8 p.p.). SP se manteve estável, entre 36,5% e 37,7%.

### Pergunta 3: quais modalidades têm maior representatividade e como a composição evoluiu?

Uso as receitas do arquivo de modalidades (`Ses_Dados_Cap`), que correspondem aos prêmios: arrecadação líquida de devoluções e cancelamentos. As modalidades 0 (sem descrição na origem) e 5 (títulos antigos, não adequados à regulamentação atual) ficam fora da análise, e o peso delas é informado no gráfico. À esquerda, a participação de cada modalidade nas receitas de cada ano; à direita, o valor em R$ bilhões, para separar mudança de composição de crescimento.

In [0]:
import matplotlib.pyplot as plt

# 1. Consulta: receitas por ano e modalidade
df_mod = spark.sql("""
    SELECT t.ano, m.cod_modalidade, d.modalidade, SUM(m.receitas) AS receitas
    FROM susep_capitalizacao.gold.fato_modalidade m
    JOIN susep_capitalizacao.gold.dim_tempo      t ON t.damesano       = m.damesano
    JOIN susep_capitalizacao.gold.dim_modalidade d ON d.cod_modalidade = m.cod_modalidade
    GROUP BY t.ano, m.cod_modalidade, d.modalidade
""").toPandas()
df_mod["receitas"] = df_mod["receitas"].astype(float)

# 2. Recorte: modalidades 0 e 5 fora da análise, com o peso informado
excluidas = df_mod["cod_modalidade"].isin([0, 5])
peso_excluidas = 100 * df_mod.loc[excluidas, "receitas"].sum() / df_mod["receitas"].sum()
peso_txt = "menos de 0.1%" if abs(peso_excluidas) < 0.1 else f"{peso_excluidas:.1f}%"
base = df_mod[~excluidas].copy()
base["pct"] = 100 * base["receitas"] / base.groupby("ano")["receitas"].transform("sum")

# Cor fixa e nome curto por modalidade (a cor segue a modalidade, não a posição)
cores  = {1: "#2a78d6", 3: "#eb6834", 4: "#1baf7a", 6: "#eda100", 7: "#e87ba4"}
curtos = {1: "Tradicional", 3: "Popular", 4: "Incentivo", 6: "Filantropia", 7: "Garantia"}
abrev  = {1: "Trad.", 3: "Pop.", 4: "Inc.", 6: "Filant.", 7: "Gar."}
nomes  = base.drop_duplicates("cod_modalidade").set_index("cod_modalidade")["modalidade"]
ordem  = base.groupby("cod_modalidade")["receitas"].sum().sort_values(ascending=False).index

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.15, 1]})
fig.patch.set_facecolor("#fcfcfb")

# 3. Esquerda: participação por modalidade, ano a ano (barras 100% empilhadas)
comp = base.pivot_table(index="ano", columns="cod_modalidade", values="pct")[ordem].sort_index(ascending=False)
esquerda = [0] * len(comp)
for cod in ordem:
    valores = comp[cod].values
    ax1.barh(comp.index.astype(str), valores, left=esquerda, color=cores[cod],
             edgecolor="#fcfcfb", linewidth=1.5, height=0.65, label=nomes[cod])
    for i, v in enumerate(valores):
        if v >= 5:                                  # fatia larga: rótulo dentro
            ax1.text(esquerda[i] + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white")
    esquerda = [e + v for e, v in zip(esquerda, valores)]

# Fatias menores que 5%: rótulo à direita da barra, com nome abreviado
for i, ano in enumerate(comp.index):
    pequenas = [f"{abrev[c]} {comp.loc[ano, c]:.0f}%" for c in ordem if comp.loc[ano, c] < 5]
    ax1.text(100.8, i, " · ".join(pequenas), ha="left", va="center", fontsize=8, color="#333")

ax1.set_xlim(0, 122)
ax1.set_xticks(range(0, 101, 20))
ax1.set_xlabel("% das receitas do ano")
ax1.set_title("Participação por modalidade", loc="left", fontsize=11)
ax1.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.42, -0.15), ncol=3, fontsize=9)

# 4. Direita: receitas por modalidade em R$ bilhões
vol = base.pivot_table(index="ano", columns="cod_modalidade", values="receitas")[ordem] / 1e9
x_fim = vol.index.max()
for cod in ordem:
    ax2.plot(vol.index, vol[cod], color=cores[cod], linewidth=2, marker="o", markersize=5)

# Rótulos finais com nome e valor, afastados quando ficam muito próximos
posicoes = []
for valor, cod in sorted((vol[c].iloc[-1], c) for c in ordem):
    y = valor if not posicoes or valor - posicoes[-1] >= 1.0 else posicoes[-1] + 1.0
    posicoes.append(y)
    ax2.annotate(f"{curtos[cod]} {valor:.1f}", xy=(x_fim, valor), xytext=(x_fim + 0.15, y),
                 textcoords="data", va="center", fontsize=9, color="#333")

ax2.set_xticks(vol.index)
ax2.set_xlim(vol.index.min() - 0.3, x_fim + 1.3)
ax2.set_ylabel("R$ bilhões")
ax2.set_title("Receitas por modalidade (R$ bi)", loc="left", fontsize=11)
ax2.grid(axis="y", color="#e5e5e5", linewidth=0.8)

for ax in (ax1, ax2):
    ax.set_facecolor("#fcfcfb")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Pergunta 3: composição das receitas por modalidade, 2021–2025 "
             f"(modalidades 0 e 5 excluídas: {peso_txt} das receitas do período)",
             x=0.01, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

#### Discussão da pergunta 3

| Modalidade | Participação 2021 | Participação 2025 | Receitas 2021 (R$ bi) | Receitas 2025 (R$ bi) |
|---|---|---|---|---|
| Tradicional | 71% | 72% | 17,1 | 24,3 |
| Filantropia Premiável | 13% | 12% | 3,1 | 4,1 |
| Instrumento de Garantia | 12% | 12% | 2,9 | 4,0 |
| Incentivo | 3% | 4% | 0,8 | 1,3 |
| Popular | 1% | 1% | 0,3 | 0,3 |

**A modalidade Tradicional domina o mercado.** Ela responde por cerca de três quartos das receitas em todos os anos (entre 71% e 74%). Filantropia Premiável e Instrumento de Garantia vêm em seguida, com 10% a 13% cada, e Incentivo e Popular somam menos de 5%. As modalidades excluídas (0 e 5) representam menos de 0,1% das receitas do período, o que confirma que o recorte não afeta o resultado.

**A composição se manteve estável no período.** A ordem das modalidades é a mesma em todos os anos, e as participações variam poucos pontos percentuais. A Tradicional atingiu o pico em 2022 (74%) e voltou a 72% em 2025. Instrumento de Garantia recuou para 10% em 2023 e 2024 e retornou a 12% em 2025. Incentivo passou de 3% para 4%, e Popular ficou em torno de 1% em todos os anos.

### Pergunta 4: a receita de cada empresa se concentra em poucas modalidades? Quem lidera cada modalidade?

Quero saber se a receita de cada empresa se concentra em uma modalidade ou se distribui entre várias, e quem lidera cada modalidade. Uso a soma das receitas de 2021 a 2025 da `fato_modalidade`, com o mesmo recorte da pergunta 3 (sem os códigos 0 e 5).

Calculo dois percentuais para cada par empresa × modalidade:

| Percentual | Pergunta que responde | Soma 100% em |
|---|---|---|
| `pct_na_modalidade` | Quanto a empresa representa dentro da modalidade? (quem lidera) | Cada modalidade |
| `pct_na_empresa` | Quanto a modalidade pesa na receita da empresa? (concentração da carteira) | Cada empresa |

As duas contas usam a mesma função de janela `SUM() OVER`; só muda o `PARTITION BY`, por modalidade ou por empresa. Deixo de fora os pares com receita total negativa ou zero no período (`HAVING`), porque um valor negativo distorceria as participações.

Salvo o resultado numa view temporária, `p4_empresa_modalidade`, que alimenta o gráfico abaixo, sem imprimir as 60 linhas do detalhe.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW p4_empresa_modalidade AS
WITH base AS (
  SELECT d.modalidade, e.nome_empresa, SUM(m.receitas) AS receitas
  FROM susep_capitalizacao.gold.fato_modalidade m
  JOIN susep_capitalizacao.gold.dim_modalidade d ON d.cod_modalidade = m.cod_modalidade
  JOIN susep_capitalizacao.gold.dim_empresa e    ON e.coenti = m.coenti
  WHERE m.cod_modalidade NOT IN (0, 5)
  GROUP BY d.modalidade, e.nome_empresa
  HAVING SUM(m.receitas) > 0
)
SELECT
  modalidade,
  nome_empresa,
  receitas,
  ROUND(100 * receitas / SUM(receitas) OVER (PARTITION BY modalidade), 1)   AS pct_na_modalidade,
  ROUND(100 * receitas / SUM(receitas) OVER (PARTITION BY nome_empresa), 1) AS pct_na_empresa,
  ROW_NUMBER() OVER (PARTITION BY modalidade ORDER BY receitas DESC)       AS posicao
FROM base

Para visualizar, monto dois mapas de calor lado a lado, com as empresas nas linhas (da maior para a menor em receitas) e as modalidades nas colunas. No da esquerda, cada coluna soma 100% e mostra quem lidera cada modalidade. No da direita, cada linha soma 100% e mostra o mix de cada empresa. Célula vazia = a empresa não atua na modalidade; "<1" = atua, mas com menos de 1%.

In [0]:
import re
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

df = spark.table("p4_empresa_modalidade").toPandas()
for col in ["receitas", "pct_na_modalidade", "pct_na_empresa"]:
    df[col] = df[col].astype(float)

def nome_curto(nome):
    curto = re.sub(r"CAPITALIZA[ÇC][ÃA]O|\bS\.A\.?|\bS/A\b", " ", nome)
    curto = re.sub(r"\s+", " ", curto).strip(" -")
    curto = re.sub(r"\s+DE$", "", curto)
    return curto.title() if curto else nome

df["empresa"] = df["nome_empresa"].apply(nome_curto)

ordem_emp = df.groupby("empresa")["receitas"].sum().sort_values(ascending=False).index
ordem_mod = df.groupby("modalidade")["receitas"].sum().sort_values(ascending=False).index

part = df.pivot(index="empresa", columns="modalidade", values="pct_na_modalidade").reindex(index=ordem_emp, columns=ordem_mod)
mix  = df.pivot(index="empresa", columns="modalidade", values="pct_na_empresa").reindex(index=ordem_emp, columns=ordem_mod)

cmap = LinearSegmentedColormap.from_list("azul", ["#fcfcfb", "#2a78d6"])
fig, axes = plt.subplots(1, 2, figsize=(15, 0.45 * len(ordem_emp) + 2.5), facecolor="#fcfcfb", sharey=True)
titulos = ["Participação da empresa em cada modalidade (cada coluna soma 100%)",
           "Mix de modalidades de cada empresa (cada linha soma 100%)"]

for ax, tabela, titulo in zip(axes, [part, mix], titulos):
    ax.set_facecolor("#fcfcfb")
    ax.imshow(tabela.fillna(0).values, cmap=cmap, vmin=0, vmax=100, aspect="auto")
    for i in range(tabela.shape[0]):
        for j in range(tabela.shape[1]):
            v = tabela.iat[i, j]
            if pd.isna(v):
                continue
            txt = "<1" if v < 1 else f"{v:.0f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                    color="white" if v >= 60 else "#333333")
    ax.set_xticks(range(tabela.shape[1]))
    ax.set_xticklabels([textwrap.fill(m, 12) for m in tabela.columns], fontsize=9)
    ax.set_title(titulo, fontsize=11, loc="left")
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)

axes[0].set_yticks(range(len(ordem_emp)))
axes[0].set_yticklabels(ordem_emp, fontsize=9)
fig.suptitle("Pergunta 4 — Composição das empresas por modalidade (receitas 2021–2025, %)",
             fontsize=13, fontweight="bold", x=0.01, ha="left")
plt.tight_layout()
plt.show()

#### Discussão da pergunta 4

| Modalidade | Líderes (participação na modalidade) | Empresas com 1% ou mais |
|---|---|---|
| Tradicional | Bradesco 30%, Brasilcap 27%, Itaú 14%, Santander 14% | 8 |
| Filantropia Premiável | Kovr 39%, Capemisa 30%, Aplicap 19%, Via 11% | 4 |
| Instrumento de Garantia | Porto Seguro 38%, Santander 26%, Icatu 24% | 5 |
| Incentivo | Icatu 31%, Bradesco 14%, Itaú 13% | 12 |
| Popular | Liderança 98% | 2 |

**Resposta:** a receita das empresas é concentrada. Em 14 das 17 empresas, uma única modalidade responde por 90% ou mais da receita (painel da direita). Só três têm a receita distribuída de forma relevante: Icatu (Tradicional 43%, Garantia 41%, Incentivo 16%), Santander (Tradicional 77%, Garantia 21%) e Mapfre (Garantia 65%, Incentivo 34%).

Receita concentrada não significa que a empresa atue em uma modalidade só. Várias aparecem em quase todas, mas com volume pequeno fora da predominante, que pode refletir o canal de distribuição e não a estratégia da empresa. Os dados da SUSEP mostram onde está a receita, não as iniciativas comerciais de cada companhia.

**Quem lidera:** cada modalidade tem o seu próprio grupo de líderes, e os grupos quase não se misturam. As quatro líderes da Tradicional pertencem a grupos bancários e somam 85% da modalidade. Nenhuma delas tem participação relevante na Filantropia Premiável, onde quatro empresas (Kovr, Capemisa, Aplicap e Via) somam 99%. O Instrumento de Garantia é liderado pela Porto Seguro, a Popular é praticamente exclusiva da Liderança (98%), e o Incentivo é a modalidade mais disputada, com 12 empresas acima de 1% e líder com 31%.

**Relação com a pergunta 1:** no mercado como um todo, os prêmios não são concentrados pelo HHI. Por modalidade, o quadro muda: como a receita de cada empresa se concentra em poucas modalidades, a disputa acontece dentro de cada uma, entre poucos participantes relevantes. Para o posicionamento de portfólio, crescer numa modalidade significa enfrentar um grupo pequeno e já estabelecido de líderes, e não o mercado inteiro.

### Conclusão

O objetivo era entender a estrutura do mercado de capitalização de 2021 a 2025: quem concentra, onde e com quais produtos. Os dados mostram um mercado estável, dominado por um produto e por uma região, em que a competição se organiza por modalidade.

| Eixo | Resposta |
|---|---|
| **Quem** (P1) | Poucas empresas lideram. As 3 maiores detêm mais da metade dos prêmios, e as 5 maiores, de 70% a 75%. Pelo HHI, o mercado de prêmios não é concentrado, mas o de provisões é moderadamente concentrado, com as 5 maiores somando cerca de 80%. A Bradesco lidera em prêmios, e a Brasilcap em provisões desde 2022 |
| **Onde** (P2) | O Sudeste domina, com mais da metade dos prêmios (55% a 58%), e São Paulo sozinho responde por mais de um terço. Sudeste e Sul somam mais de 70%, e as outras três regiões juntas ficam em cerca de um quarto. O quadro se manteve ao longo do período |
| **Com quais produtos** (P3) | A Tradicional domina, com cerca de 72% das receitas. Filantropia Premiável e Instrumento de Garantia vêm em seguida, com cerca de 12% cada, e Incentivo e Popular somam menos de 5%. A composição se manteve estável |
| **Quem lidera cada produto** (P4) | Cada modalidade tem o seu próprio grupo de líderes: na Tradicional, Bradesco, Brasilcap, Itaú e Santander somam 85%; na Filantropia Premiável, Kovr, Capemisa, Aplicap e Via somam 99%; no Instrumento de Garantia, Porto Seguro, Santander e Icatu somam 88%; e na Popular, a Liderança tem 98%. Em 14 das 17 empresas, uma única modalidade responde por 90% ou mais da receita |

**Visão integrada.** Os quatro recortes contam a mesma história: o mercado gira em torno do título Tradicional, liderado por empresas de grupos bancários, com a demanda concentrada no Sudeste. O HHI de prêmios, abaixo de 1.500, não capta isso sozinho. A concentração aparece quando se olha por produto, porque cada modalidade é disputada por poucas empresas. O quadro também é estável: em cinco anos, a liderança, a distribuição regional e a composição dos produtos mudaram pouco.

**Para o planejamento**, a posição competitiva deve ser avaliada por modalidade, e não só no mercado total, porque é dentro de cada modalidade que a disputa acontece.